# Earth vs Mars: the Grand Canyon as a terrestrial analog

Reproduces the comparison of tributary (side) canyons in the Grand Canyon (Earth) and Valles Marineris (Mars).
All rules are written in units of the main-canyon depth **D**, calibrated on Earth only, then frozen and applied to Mars.

**Inputs:** USGS 3DEP 1 arc-second (Microsoft Planetary Computer), USGS HRSC–MOLA Blended DEM 200 m v2 (read remotely), and the YOLO catalogue in `results/grand_canyon/`.
**Outputs:** `results/mars_analog/`. Large intermediate rasters are ignored by git.

Run the cells in order. On Google Colab (High-RAM recommended): clone the repository, `pip install -r requirements.txt`, then open this notebook from the `notebooks/` folder.

In [ ]:
from pathlib import Path

REPO_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
(REPO_DIR / 'results/mars_analog').mkdir(parents=True, exist_ok=True)
print('Repository:', REPO_DIR)

## B1 – Two comparable 200 m elevation models (Earth: USGS 3DEP, Mars: HRSC–MOLA)

In [ ]:
# Step B1: build two comparable elevation models at the same 200 m pixel size
import json
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import planetary_computer
import rasterio
from pystac_client import Client
from rasterio.enums import Resampling
from rasterio.transform import from_origin
from rasterio.vrt import WarpedVRT
from rasterio.warp import transform_bounds
from rasterio.windows import from_bounds
from scipy.ndimage import zoom


PROJECT_DIR = REPO_DIR
OUT_DIR = PROJECT_DIR / 'results/mars_analog'
PIXEL_M = 200

# Earth: whole Grand Canyon region, USGS 3DEP 1 arc-second (about 30 m), projected to UTM 12N
GC_BOUNDS = (-114.25, 35.45, -111.25, 37.05)
GC_CRS = 'EPSG:32612'
GC_FILE = OUT_DIR / 'grand_canyon_dem_200m.tif'

# Mars: Valles Marineris, USGS HRSC-MOLA blended DEM (200 m), converted to metres
MARS_URL = '/vsicurl/https://planetarymaps.usgs.gov/mosaic/Mars/HRSC_MOLA_Blend/Mars_HRSC_MOLA_BlendDEM_Global_200mp_v2.tif'
VM_BOUNDS = (-100.0, -18.0, -30.0, 0.0)
MARS_RADIUS_M = 3396190
STANDARD_LAT = -8.0  # centre latitude used to turn degrees into metres
VM_FILE = OUT_DIR / 'valles_marineris_dem_200m.tif'

OUT_DIR.mkdir(parents=True, exist_ok=True)


def build_grand_canyon():
    catalog = Client.open('https://planetarycomputer.microsoft.com/api/stac/v1', modifier=planetary_computer.sign_inplace)
    items = [item for item in catalog.search(collections=['3dep-seamless'], bbox=GC_BOUNDS).items()
             if item.properties.get('gsd') == 30]
    if not items:
        raise ValueError('No 3DEP 30 m tiles were found.')
    left, bottom, right, top = transform_bounds('EPSG:4326', GC_CRS, *GC_BOUNDS, densify_pts=21)
    width = int(np.ceil((right - left) / PIXEL_M))
    height = int(np.ceil((top - bottom) / PIXEL_M))
    transform = from_origin(left, top, PIXEL_M, PIXEL_M)
    dem = np.full((height, width), np.nan, dtype='float32')
    for number, item in enumerate(items, start=1):
        print(f'  3DEP tile {number}/{len(items)}: {item.id}')
        with rasterio.open(item.assets['data'].href) as src:
            with WarpedVRT(src, crs=GC_CRS, transform=transform, width=width, height=height,
                           resampling=Resampling.average) as vrt:
                tile = vrt.read(1).astype('float32')
        tile[(tile <= -100) | (tile > 9000)] = np.nan
        dem = np.where(np.isnan(dem), tile, dem)
    profile = dict(driver='GTiff', width=width, height=height, count=1, dtype='float32', crs=GC_CRS,
                   transform=transform, nodata=np.nan, tiled=True, compress='deflate')
    with rasterio.open(GC_FILE, 'w', **profile) as dst:
        dst.write(dem, 1)


def build_valles_marineris():
    print('  Reading Valles Marineris from USGS (about 1.2 GB transfer)...')
    with rasterio.Env(GDAL_DISABLE_READDIR_ON_OPEN='EMPTY_DIR', CPL_VSIL_CURL_ALLOWED_EXTENSIONS='.tif'):
        with rasterio.open(MARS_URL) as src:
            window = from_bounds(*VM_BOUNDS, transform=src.transform).round_offsets().round_lengths()
            data = src.read(1, window=window).astype('float32')
            data[data == src.nodata] = np.nan
            window_transform = src.window_transform(window)
            degree_size = src.res[0]
    metres_per_degree = np.pi / 180 * MARS_RADIUS_M
    x_size = degree_size * metres_per_degree * np.cos(np.radians(STANDARD_LAT))
    y_size = degree_size * metres_per_degree
    dem = zoom(data, (y_size / PIXEL_M, x_size / PIXEL_M), order=1)
    left = window_transform.c * metres_per_degree * np.cos(np.radians(STANDARD_LAT))
    top = window_transform.f * metres_per_degree
    profile = dict(driver='GTiff', width=dem.shape[1], height=dem.shape[0], count=1, dtype='float32',
                   transform=from_origin(left, top, PIXEL_M, PIXEL_M), nodata=np.nan, tiled=True, compress='deflate')
    with rasterio.open(VM_FILE, 'w', **profile) as dst:
        dst.write(dem.astype('float32'), 1)
    (OUT_DIR / 'valles_marineris_grid.json').write_text(json.dumps({
        'source': 'USGS HRSC-MOLA Blended DEM Global 200m v2 (Fergason et al., 2018)',
        'bounds_degrees': VM_BOUNDS, 'projection': 'equirectangular, Mars sphere R=3396190 m',
        'standard_latitude': STANDARD_LAT, 'pixel_m': PIXEL_M,
        'x_scale_error_percent': round(float((1 - np.cos(np.radians(18)) / np.cos(np.radians(8))) * 100), 1),
    }, indent=2))


for name, path, builder in [('Grand Canyon', GC_FILE, build_grand_canyon),
                            ('Valles Marineris', VM_FILE, build_valles_marineris)]:
    if path.exists():
        print(f'{name}: using saved file')
    else:
        start = time.time()
        print(f'{name}: building...')
        builder()
        print(f'{name}: saved in {time.time() - start:.0f} s')


def summary(name, path):
    with rasterio.open(path) as src:
        dem = src.read(1)
    valid = dem[~np.isnan(dem)]
    km = (dem.shape[1] * PIXEL_M / 1000, dem.shape[0] * PIXEL_M / 1000)
    print(f'{name:<17} grid {dem.shape[1]} x {dem.shape[0]} px = {km[0]:,.0f} x {km[1]:,.0f} km | '
          f'elevation {valid.min():,.0f} to {valid.max():,.0f} m | relief {valid.max() - valid.min():,.0f} m | '
          f'missing {np.isnan(dem).mean() * 100:.1f}%')
    return dem, km


print('\n===== Comparable elevation models (200 m pixels) =====')
gc_dem, gc_km = summary('Grand Canyon', GC_FILE)
vm_dem, vm_km = summary('Valles Marineris', VM_FILE)


def shaded(dem):
    dy, dx = np.gradient(np.nan_to_num(dem, nan=np.nanmedian(dem)), PIXEL_M)
    slope = np.arctan(np.hypot(dx, dy))
    aspect = np.arctan2(-dx, dy)
    return np.clip(np.cos(slope) * 0.7 + np.sin(slope) * np.cos(np.radians(315) - aspect) * 0.7, 0, 1)


step = 4
fig, axes = plt.subplots(2, 1, figsize=(20, 13), gridspec_kw={'height_ratios': [1, 1.25]})
axes[0].imshow(shaded(vm_dem[::step, ::step]), cmap='gray', extent=(0, vm_km[0], 0, vm_km[1]))
axes[0].imshow(vm_dem[::step, ::step], cmap='terrain', alpha=0.45, extent=(0, vm_km[0], 0, vm_km[1]))
axes[0].add_patch(plt.Rectangle((40, 40), gc_km[0], gc_km[1], fill=False, edgecolor='white', linewidth=2))
axes[0].text(40, 40 + gc_km[1] + 15, 'Grand Canyon region at the same scale', color='white', fontsize=11)
axes[0].set_title('Valles Marineris, Mars (200 m)')
axes[1].imshow(shaded(gc_dem), cmap='gray', extent=(0, gc_km[0], 0, gc_km[1]))
axes[1].imshow(gc_dem, cmap='terrain', alpha=0.45, extent=(0, gc_km[0], 0, gc_km[1]))
axes[1].set_title('Grand Canyon region, Earth (200 m)')
for axis in axes:
    axis.set_xlabel('km')
    axis.set_ylabel('km')
plt.tight_layout()
plt.savefig(OUT_DIR / 'b1_same_pixel_size.png', dpi=100)
plt.show()
print('Saved to:', OUT_DIR)


## B2a – Incision depth below the plateau (main-canyon depth D)

In [ ]:
# Step B2a: incision depth - how far each pixel lies below the surrounding plateau
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import rasterio
from scipy.ndimage import gaussian_filter, maximum_filter, minimum_filter, zoom


OUT_DIR = REPO_DIR / 'results/mars_analog'
PIXEL_M = 200
# The window must be wider than the main canyon, so the plateau on both sides is included
REGIONS = {
    'Grand Canyon': {'dem': OUT_DIR / 'grand_canyon_dem_200m.tif', 'window_km': 30},
    'Valles Marineris': {'dem': OUT_DIR / 'valles_marineris_dem_200m.tif', 'window_km': 400},
}


def incision_depth(dem, window_px):
    factor = max(1, window_px // 100)  # work on a coarser copy for very wide windows
    small = maximum_filter(dem, size=factor)[::factor, ::factor] if factor > 1 else dem
    size = (window_px // factor) | 1
    plateau = minimum_filter(maximum_filter(small, size=size), size=size)  # grey closing
    plateau = gaussian_filter(plateau, sigma=size / 8)
    if factor > 1:
        plateau = zoom(plateau, (dem.shape[0] / plateau.shape[0], dem.shape[1] / plateau.shape[1]), order=1)
        plateau = np.pad(plateau, [(0, max(0, dem.shape[0] - plateau.shape[0])), (0, max(0, dem.shape[1] - plateau.shape[1]))],
                         mode='edge')[:dem.shape[0], :dem.shape[1]]
    return np.clip(plateau - dem, 0, None).astype('float32')


results = {}
for name, settings in REGIONS.items():
    if not settings['dem'].exists():
        raise FileNotFoundError(f"Missing file: {settings['dem']} - run Step B1 first")
    with rasterio.open(settings['dem']) as src:
        dem = src.read(1)
        profile = src.profile
    window_px = int(settings['window_km'] * 1000 / PIXEL_M) | 1
    print(f'{name}: computing incision depth (window {settings["window_km"]} km = {window_px} px)...')
    depth = incision_depth(dem, window_px)
    out_file = OUT_DIR / f"{name.lower().replace(' ', '_')}_incision_depth.tif"
    with rasterio.open(out_file, 'w', **profile) as dst:
        dst.write(depth, 1)
    reference = float(np.percentile(depth, 99.5))
    results[name] = {
        'depth': depth,
        'reference_m': reference,
        'deep_share': float((depth >= 0.25 * reference).mean() * 100),
        'max_m': float(depth.max()),
    }

print('\n===== Incision depth =====')
for name, item in results.items():
    print(f"{name:<17} main-canyon depth (99.5th percentile) {item['reference_m']:,.0f} m | "
          f"max {item['max_m']:,.0f} m | area deeper than 25% of that: {item['deep_share']:.1f}%")
ratio = results['Valles Marineris']['reference_m'] / results['Grand Canyon']['reference_m']
print(f'Depth ratio (Mars / Earth): {ratio:.1f} x')

fig, axes = plt.subplots(2, 1, figsize=(20, 12), gridspec_kw={'height_ratios': [1, 1.25]})
for axis, (name, item), step in zip(axes, results.items(), [8, 1]):
    shown = item['depth'][::step, ::step] / item['reference_m']
    km = (item['depth'].shape[1] * PIXEL_M / 1000, item['depth'].shape[0] * PIXEL_M / 1000)
    image = axis.imshow(np.clip(shown, 0, 1), cmap='magma', extent=(0, km[0], 0, km[1]))
    axis.set_title(f"{name}: depth below plateau / main-canyon depth ({item['reference_m']:,.0f} m)")
    axis.set_xlabel('km')
    axis.set_ylabel('km')
    fig.colorbar(image, ax=axis, fraction=0.02, label='relative depth (0 = plateau, 1 = deepest)')
plt.tight_layout()
plt.savefig(OUT_DIR / 'b2a_incision_depth.png', dpi=100)
plt.show()
print('Saved to:', OUT_DIR)


## B2a – Plot the incision-depth maps with little memory

In [ ]:
# Step B2a (plot only): draw the saved incision-depth maps using little memory
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import rasterio
from rasterio.enums import Resampling


OUT_DIR = REPO_DIR / 'results/mars_analog'
PIXEL_M = 200
MAPS = {
    'Grand Canyon': (OUT_DIR / 'grand_canyon_incision_depth.tif', 1, 1229),
    'Valles Marineris': (OUT_DIR / 'valles_marineris_incision_depth.tif', 8, 9237),
}

fig, axes = plt.subplots(2, 1, figsize=(20, 12), gridspec_kw={'height_ratios': [1, 1.25]})
for axis, (name, (path, step, reference)) in zip(axes, MAPS.items()):
    if not path.exists():
        raise FileNotFoundError(f'Missing file: {path}')
    with rasterio.open(path) as src:
        depth = src.read(1, out_shape=(src.height // step, src.width // step), resampling=Resampling.average)
        km = (src.width * PIXEL_M / 1000, src.height * PIXEL_M / 1000)
    image = axis.imshow(np.clip(depth / reference, 0, 1), cmap='magma', extent=(0, km[0], 0, km[1]))
    axis.set_title(f'{name}: depth below plateau / main-canyon depth ({reference:,} m)')
    axis.set_xlabel('km')
    axis.set_ylabel('km')
    fig.colorbar(image, ax=axis, fraction=0.02, label='relative depth (0 = plateau, 1 = deepest)')
    print(f'{name}: map read at {depth.shape[1]} x {depth.shape[0]} px')
plt.tight_layout()
plt.savefig(OUT_DIR / 'b2a_incision_depth.png', dpi=100)
plt.show()
print('Saved to:', OUT_DIR)


## B3c – Earth calibration: rim-rule sensitivity against the YOLO catalogue

In [ ]:
# Step B3c: keep only the incised part of each tributary - sensitivity test on Earth
import sys
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
from rasterio.features import rasterize, shapes
from scipy.ndimage import label, maximum_filter, minimum_filter, uniform_filter
from shapely.geometry import shape


PROJECT_DIR = REPO_DIR
OUT_DIR = PROJECT_DIR / 'results/mars_analog'
TOOLS_DIR = PROJECT_DIR / 'analog'
CATALOGUE = PROJECT_DIR / 'results/grand_canyon/grand_canyon_detections.gpkg'
PIXEL_M = 200
D = 1229
RULES = {'trunk_depth_D': 0.5, 'floor_half_width_D': 0.5, 'tributary_window_D': 4.0,
         'min_incision_D': 0.15, 'min_area_D2': 1.0}
RIM_OPTIONS_D = [0.05, 0.10, 0.15, 0.20]  # a pixel is "inside" a tributary if it lies this far below the local rim

if not (TOOLS_DIR / 'flowtools.py').exists():
    raise FileNotFoundError('Missing analog/flowtools.py - run Step B3b first')
sys.path.insert(0, str(TOOLS_DIR))
import flowtools


def closing_depth(dem, window_px):
    size = window_px | 1
    surface = minimum_filter(maximum_filter(dem, size=size), size=size)
    surface = uniform_filter(surface, size=max(3, size // 4))
    return np.clip(surface - dem, 0, None)


def share(a, b):
    return (a & b).sum() / max(a.sum(), 1)


with rasterio.open(OUT_DIR / 'grand_canyon_dem_200m.tif') as src:
    dem = src.read(1).astype('float64')
    transform, crs = src.transform, src.crs
with rasterio.open(OUT_DIR / 'grand_canyon_incision_depth.tif') as src:
    main_depth = src.read(1)
catalogue = gpd.read_file(CATALOGUE).to_crs('EPSG:32612')
zone = rasterize([(catalogue.buffer(5000).union_all(), 1)], out_shape=dem.shape, transform=transform, dtype='uint8').astype(bool)
reference = rasterize([(g, 1) for g in catalogue.geometry], out_shape=dem.shape, transform=transform, dtype='uint8').astype(bool)
masks = [rasterize([(g, 1)], out_shape=dem.shape, transform=transform, dtype='uint8').astype(bool) for g in catalogue.geometry]
masks = [m for m in masks if m.sum()]

print('Routing flow (same as B3b)...')
filled = flowtools.fill_depressions(dem)
receiver = flowtools.flow_receivers(filled)
acc = flowtools.accumulation(filled, receiver)
trunk_raw = maximum_filter(minimum_filter(((main_depth >= RULES['trunk_depth_D'] * D) & zone).astype(np.uint8), size=5), size=5).astype(bool)
labels, _ = label(trunk_raw)
sizes = np.bincount(labels.ravel())
sizes[0] = 0
trunk = labels == sizes.argmax()
stem = flowtools.main_stem(acc, receiver, trunk)
floor = maximum_filter(stem.astype(np.uint8), size=2 * round(RULES['floor_half_width_D'] * D / PIXEL_M) + 1).astype(bool)
entry = flowtools.entry_points(filled, receiver, floor)
local = closing_depth(dem, int(RULES['tributary_window_D'] * D / PIXEL_M))


def build(rim_D):
    canyon = ((local >= rim_D * D) | (main_depth >= RULES['trunk_depth_D'] * D)) & zone & ~floor & (entry >= 0)
    ids, inverse = np.unique(entry[canyon], return_inverse=True)
    area = np.bincount(inverse) * PIXEL_M ** 2
    incision = np.zeros(len(ids))
    np.maximum.at(incision, inverse, local[canyon])
    keep = (area >= RULES['min_area_D2'] * D * D) & (incision >= RULES['min_incision_D'] * D)
    new_id = np.where(keep, np.cumsum(keep), 0)
    result = np.zeros(dem.shape, dtype='int32')
    result[canyon] = new_id[inverse]
    return result


rows = []
outputs = {}
for rim in RIM_OPTIONS_D:
    tributaries = build(rim)
    tributary = tributaries > 0
    precision = share(tributary, reference)
    recall = share(reference, tributary)
    covered = np.array([share(m, tributary) for m in masks])
    areas = np.bincount(tributaries.ravel())[1:] * PIXEL_M ** 2 / D ** 2
    rows.append({'rim_depth_D': rim, 'tributaries': int(tributaries.max()),
                 'area_precision_%': round(precision * 100), 'area_recall_%': round(recall * 100),
                 'F1_%': round(2 * precision * recall / max(precision + recall, 1e-9) * 100),
                 'canyons_covered_50%': round(np.mean(covered >= 0.5) * 100),
                 'median_area_D2': round(float(np.median(areas)), 1)})
    outputs[rim] = tributaries

results = pd.DataFrame(rows)
best = results.loc[results['F1_%'].idxmax(), 'rim_depth_D']
results.to_csv(OUT_DIR / 'b3c_sensitivity.csv', index=False)
print('\n===== Sensitivity of the rim rule (Earth calibration) =====')
print('B3b (no rim rule): 264 tributaries | precision 31% | recall 78% | covered 91%')
print(results.to_string(index=False))
print(f'Best F1: rim depth = {best} D ({best * D:.0f} m)')

tributaries = outputs[best]
records = [{'tributary_id': int(v), 'geometry': shape(g)} for g, v in shapes(tributaries, mask=tributaries > 0, transform=transform)]
table = gpd.GeoDataFrame(records, geometry='geometry', crs=crs).dissolve(by='tributary_id').reset_index()
table['area_km2'] = table.area / 1e6
table['area_D2'] = table.area / D ** 2
table['rim_depth_D'] = best
table.to_file(OUT_DIR / 'grand_canyon_tributaries_final.gpkg', driver='GPKG')

r, c = np.where(zone)
r0, r1, c0, c1 = r.min(), r.max() + 1, c.min(), c.max() + 1
extent = (c0 * PIXEL_M / 1000, c1 * PIXEL_M / 1000, (dem.shape[0] - r1) * PIXEL_M / 1000, (dem.shape[0] - r0) * PIXEL_M / 1000)
colours = np.random.default_rng(1).uniform(0.3, 1.0, size=(tributaries.max() + 1, 3))
overlay = np.zeros((r1 - r0, c1 - c0, 4))
part = tributaries[r0:r1, c0:c1]
overlay[part > 0, :3] = colours[part[part > 0]]
overlay[part > 0, 3] = 0.75
overlay[floor[r0:r1, c0:c1]] = (0.2, 0.6, 1.0, 0.8)
edge = reference & ~minimum_filter(reference.astype(np.uint8), size=3).astype(bool)
overlay[edge[r0:r1, c0:c1]] = (0.0, 1.0, 0.4, 1.0)
fig, axis = plt.subplots(figsize=(18, 11))
axis.imshow(np.clip(main_depth[r0:r1, c0:c1] / D, 0, 1), cmap='gray', extent=extent)
axis.imshow(overlay, extent=extent)
axis.set_title(f'Final Earth tributaries (rim depth {best} D): {len(table)} tributaries, YOLO catalogue in green')
axis.set_xlabel('km')
axis.set_ylabel('km')
plt.tight_layout()
plt.savefig(OUT_DIR / 'b3c_final_tributaries.png', dpi=100)
plt.show()
print('Saved to:', OUT_DIR)


## B4 – Apply the frozen rules to Mars at the same relative resolution (first pass, floor 0.8 D)

In [ ]:
# Step B4: apply the frozen Earth rules to Mars at the same relative resolution
import json
import sys
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import rasterio
from affine import Affine
from rasterio.enums import Resampling
from rasterio.features import rasterize, shapes
from rasterio.windows import from_bounds
from scipy.ndimage import label, maximum_filter, minimum_filter, uniform_filter
from shapely.geometry import shape


PROJECT_DIR = REPO_DIR
OUT_DIR = PROJECT_DIR / 'results/mars_analog'
TOOLS_DIR = PROJECT_DIR / 'analog'
CATALOGUE = PROJECT_DIR / 'results/grand_canyon/grand_canyon_detections.gpkg'
D_EARTH, D_MARS = 1229, 9237
EARTH_PIXEL_M = 200
MARS_PIXEL_M = EARTH_PIXEL_M * D_MARS / D_EARTH   # same number of pixels per D on both planets
MARS_RADIUS_M = 3396190
STANDARD_LAT = -8.0

# Frozen after the Earth calibration (Step B3c). Do not change these for Mars.
RULES = {
    'trunk_depth_D': 0.5, 'floor_depth_D': 0.8, 'floor_half_width_D': 0.5, 'tributary_window_D': 4.0,
    'rim_depth_D': 0.05, 'min_incision_D': 0.15, 'min_area_D2': 1.0,
}
(OUT_DIR / 'frozen_rules.json').write_text(json.dumps(RULES, indent=2))

# Updated flow tools: the main-canyon floor can now act as base level (needed for closed basins)
FLOWTOOLS = r'''
import heapq

import numpy as np
from scipy.ndimage import maximum_filter, minimum_filter, uniform_filter

OFFSETS = [(-1, 0), (-1, 1), (0, 1), (1, 1), (1, 0), (1, -1), (0, -1), (-1, -1)]


def fill_depressions(dem, eps=1e-3, sinks=None):
    """Priority-flood (Barnes et al., 2014): water can always flow to the map edge,
    or into `sinks` (cells treated as base level, e.g. a main-canyon floor)."""
    height, width = dem.shape
    filled = dem.astype('float64').copy()
    done = np.zeros(dem.shape, dtype=bool)
    heap = []
    if sinks is not None:
        for r, c in zip(*np.nonzero(sinks)):
            done[r, c] = True
            heap.append((filled[r, c], r, c))
    for r in range(height):
        for c in (0, width - 1):
            if not done[r, c]:
                done[r, c] = True
                heap.append((filled[r, c], r, c))
    for c in range(width):
        for r in (0, height - 1):
            if not done[r, c]:
                done[r, c] = True
                heap.append((filled[r, c], r, c))
    heapq.heapify(heap)
    while heap:
        z, r, c = heapq.heappop(heap)
        for dr, dc in OFFSETS:
            nr, nc = r + dr, c + dc
            if 0 <= nr < height and 0 <= nc < width and not done[nr, nc]:
                done[nr, nc] = True
                filled[nr, nc] = max(filled[nr, nc], z + eps)
                heapq.heappush(heap, (filled[nr, nc], nr, nc))
    return filled


def flow_receivers(filled):
    """D8 steepest descent. Returns the flat index of the downstream cell (-1 = flows off the map)."""
    height, width = filled.shape
    padded = np.pad(filled, 1, constant_values=-np.inf)
    best = np.zeros(filled.shape)
    receiver = np.full(filled.shape, -1, dtype=np.int64)
    rows, cols = np.indices(filled.shape)
    for dr, dc in OFFSETS:
        neighbour = padded[1 + dr:1 + dr + height, 1 + dc:1 + dc + width]
        drop = (filled - neighbour) / np.hypot(dr, dc)
        inside = (rows + dr >= 0) & (rows + dr < height) & (cols + dc >= 0) & (cols + dc < width)
        better = (drop > best) & inside
        best[better] = drop[better]
        receiver[better] = ((rows + dr) * width + (cols + dc))[better]
    edge = np.zeros(filled.shape, dtype=bool)
    edge[0, :] = edge[-1, :] = edge[:, 0] = edge[:, -1] = True
    receiver[edge & (best <= 0)] = -1
    return receiver.ravel()


def accumulation(filled, receiver):
    order = np.argsort(filled.ravel())[::-1]  # high to low: upstream before downstream
    acc = np.ones(filled.size, dtype=np.int64)
    for cell in order:
        down = receiver[cell]
        if down >= 0:
            acc[down] += acc[cell]
    return acc


def main_stem(acc, receiver, inside_trunk):
    """Follow the largest flow upstream from the biggest outlet while inside the main canyon."""
    upstream = {}
    for cell, down in enumerate(receiver):
        if down >= 0:
            upstream.setdefault(down, []).append(cell)
    flat_trunk = inside_trunk.ravel()
    candidates = np.flatnonzero(flat_trunk)
    cell = candidates[np.argmax(acc[candidates])]
    stem = [cell]
    while True:
        options = [u for u in upstream.get(cell, []) if flat_trunk[u]]
        if not options:
            break
        cell = max(options, key=lambda u: acc[u])
        stem.append(cell)
    mask = np.zeros(acc.size, dtype=bool)
    mask[stem] = True
    return mask.reshape(inside_trunk.shape)


def entry_points(filled, receiver, floor):
    """For every cell, the first cell of its flow path that sits next to the main-canyon floor."""
    flat_floor = floor.ravel()
    entry = np.full(filled.size, -1, dtype=np.int64)
    for cell in np.argsort(filled.ravel()):  # low to high: downstream before upstream
        if flat_floor[cell]:
            continue
        down = receiver[cell]
        if down < 0:
            continue
        entry[cell] = cell if flat_floor[down] else entry[down]
    return entry.reshape(filled.shape)
'''
TOOLS_DIR.mkdir(parents=True, exist_ok=True)
(TOOLS_DIR / 'flowtools.py').write_text(FLOWTOOLS)
sys.path.insert(0, str(TOOLS_DIR))
import importlib
import flowtools
importlib.reload(flowtools)


def px(value_D, D, pixel_m):
    return max(1, int(round(value_D * D / pixel_m)))


def closing_depth(dem, window_px):
    size = window_px | 1
    surface = minimum_filter(maximum_filter(dem, size=size), size=size)
    surface = uniform_filter(surface, size=max(3, size // 4))
    return np.clip(surface - dem, 0, None)


def find_tributaries(dem, main_depth, zone, D, pixel_m):
    trunk = maximum_filter(minimum_filter(((main_depth >= RULES['trunk_depth_D'] * D) & zone).astype(np.uint8), size=5), size=5)
    labels, _ = label(trunk)
    sizes = np.bincount(labels.ravel())
    sizes[0] = 0
    trunk = labels == sizes.argmax()
    # Main-canyon floor = base level. (1) every deep floor area of the main canyon (>= D x D);
    # (2) a strip along the main channel found by a first flow pass. Part 1 matters on Mars, where the
    # chasmata are closed basins; filling them like lakes would drown the side canyons.
    deep, _ = label((main_depth >= RULES['floor_depth_D'] * D) & trunk)
    deep_area = np.bincount(deep.ravel()) * pixel_m ** 2
    deep_area[0] = 0
    floor = np.isin(deep, np.flatnonzero(deep_area >= D * D))
    first = flowtools.fill_depressions(dem)
    receiver = flowtools.flow_receivers(first)
    stem = flowtools.main_stem(flowtools.accumulation(first, receiver), receiver, trunk)
    floor |= maximum_filter(stem.astype(np.uint8), size=2 * px(RULES['floor_half_width_D'], D, pixel_m) + 1).astype(bool)
    # Second flow pass: water drains to the map edge or into the main-canyon floor
    filled = flowtools.fill_depressions(dem, sinks=floor)
    receiver = flowtools.flow_receivers(filled)
    entry = flowtools.entry_points(filled, receiver, floor)
    local = closing_depth(dem, 2 * px(RULES['tributary_window_D'] / 2, D, pixel_m) + 1)
    canyon = ((local >= RULES['rim_depth_D'] * D) | (main_depth >= RULES['trunk_depth_D'] * D)) & zone & ~floor & (entry >= 0)
    ids, inverse = np.unique(entry[canyon], return_inverse=True)
    area = np.bincount(inverse) * pixel_m ** 2
    incision = np.zeros(len(ids))
    np.maximum.at(incision, inverse, local[canyon])
    keep = (area >= RULES['min_area_D2'] * D * D) & (incision >= RULES['min_incision_D'] * D)
    new_id = np.where(keep, np.cumsum(keep), 0)
    tributaries = np.zeros(dem.shape, dtype='int32')
    tributaries[canyon] = new_id[inverse]
    return tributaries, floor


def to_table(tributaries, transform, crs, D, name):
    records = [{'tributary_id': int(v), 'geometry': shape(g)} for g, v in shapes(tributaries, mask=tributaries > 0, transform=transform)]
    table = gpd.GeoDataFrame(records, geometry='geometry', crs=crs).dissolve(by='tributary_id').reset_index()
    table['area_km2'] = table.area / 1e6
    table['area_D2'] = table.area / D ** 2
    table['planet'] = name
    return table


def share(a, b):
    return (a & b).sum() / max(a.sum(), 1)


# ---------- Earth (re-run with the final floor rule, to confirm calibration still holds) ----------
print('Earth: routing flow...')
with rasterio.open(OUT_DIR / 'grand_canyon_dem_200m.tif') as src:
    earth_dem = src.read(1).astype('float64')
    earth_transform, earth_crs = src.transform, src.crs
with rasterio.open(OUT_DIR / 'grand_canyon_incision_depth.tif') as src:
    earth_depth = src.read(1)
catalogue = gpd.read_file(CATALOGUE).to_crs('EPSG:32612')
earth_zone = rasterize([(catalogue.buffer(5000).union_all(), 1)], out_shape=earth_dem.shape, transform=earth_transform, dtype='uint8').astype(bool)
reference = rasterize([(g, 1) for g in catalogue.geometry], out_shape=earth_dem.shape, transform=earth_transform, dtype='uint8').astype(bool)
earth_trib, earth_floor = find_tributaries(earth_dem, earth_depth, earth_zone, D_EARTH, EARTH_PIXEL_M)
earth_table = to_table(earth_trib, earth_transform, earth_crs, D_EARTH, 'Earth')
earth_table.to_file(OUT_DIR / 'earth_tributaries_frozen.gpkg', driver='GPKG')
print(f'Earth check: {len(earth_table)} tributaries | precision {share(earth_trib > 0, reference) * 100:.0f}% | '
      f'recall {share(reference, earth_trib > 0) * 100:.0f}% | median {earth_table.area_D2.median():.1f} D2 '
      f'(B3c at 0.05 D: 233, 39%, 76%, 3.6 D2)')

# ---------- Mars: central chasmata, resampled so 1 D spans the same number of pixels as on Earth ----------
print(f'Mars: resampling to {MARS_PIXEL_M:,.0f} m pixels and routing flow...')
metres_per_degree = np.pi / 180 * MARS_RADIUS_M
mars_x = lambda lon: lon * metres_per_degree * np.cos(np.radians(STANDARD_LAT))
bounds = (mars_x(-92.0), -18.0 * metres_per_degree, mars_x(-52.0), 0.0)
scale = MARS_PIXEL_M / 200
rasters = {}
for key, file_name in [('dem', 'valles_marineris_dem_200m.tif'), ('depth', 'valles_marineris_incision_depth.tif')]:
    with rasterio.open(OUT_DIR / file_name) as src:
        window = from_bounds(*bounds, transform=src.transform).round_offsets().round_lengths()
        shape_out = (int(window.height / scale), int(window.width / scale))
        rasters[key] = src.read(1, window=window, out_shape=shape_out, resampling=Resampling.average).astype('float64')
        mars_transform = src.window_transform(window) * Affine.scale(window.width / shape_out[1], window.height / shape_out[0])
mars_zone = np.ones(rasters['dem'].shape, dtype=bool)
mars_trib, mars_floor = find_tributaries(rasters['dem'], rasters['depth'], mars_zone, D_MARS, MARS_PIXEL_M)
mars_table = to_table(mars_trib, mars_transform, None, D_MARS, 'Mars')
mars_table.to_file(OUT_DIR / 'mars_tributaries_frozen.gpkg', driver='GPKG')

print('\n===== Same rules, same relative resolution =====')
print(f'Pixel size: Earth {EARTH_PIXEL_M} m, Mars {MARS_PIXEL_M:,.0f} m  (both = D / {D_EARTH / EARTH_PIXEL_M:.1f})')
for table in (earth_table, mars_table):
    print(f"{table.planet.iloc[0]:<6} tributaries {len(table):>4} | median area {table.area_D2.median():.1f} D2 "
          f"(IQR {table.area_D2.quantile(.25):.1f}-{table.area_D2.quantile(.75):.1f}) | "
          f"median {table.area_km2.median():,.1f} km2")

fig, axes = plt.subplots(2, 1, figsize=(18, 14))
for axis, (name, trib, floor, depth, D, pixel_m) in zip(axes, [
        ('Earth - Grand Canyon', earth_trib, earth_floor, earth_depth, D_EARTH, EARTH_PIXEL_M),
        ('Mars - Valles Marineris (92W-52W)', mars_trib, mars_floor, rasters['depth'], D_MARS, MARS_PIXEL_M)]):
    extent = (0, trib.shape[1] * pixel_m / 1000, 0, trib.shape[0] * pixel_m / 1000)
    colours = np.random.default_rng(1).uniform(0.3, 1.0, size=(trib.max() + 1, 3))
    overlay = np.zeros(trib.shape + (4,))
    overlay[trib > 0, :3] = colours[trib[trib > 0]]
    overlay[trib > 0, 3] = 0.8
    overlay[floor] = (0.2, 0.6, 1.0, 0.6)
    axis.imshow(np.clip(depth / D, 0, 1), cmap='gray', extent=extent)
    axis.imshow(overlay, extent=extent)
    axis.set_title(f'{name}: {trib.max()} tributary canyons (colours), main-canyon floor (blue)')
    axis.set_xlabel('km')
    axis.set_ylabel('km')
plt.tight_layout()
plt.savefig(OUT_DIR / 'b4_earth_mars_tributaries.png', dpi=100)
plt.show()
print('Saved to:', OUT_DIR)


## B4b – Calibrate the floor rule on Earth and apply the chosen value to Mars

In [ ]:
# Step B4b: calibrate the new floor rule on Earth, then apply the chosen value to Mars
import json
import sys
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
import rasterio
from affine import Affine
from rasterio.enums import Resampling
from rasterio.features import rasterize, shapes
from rasterio.windows import from_bounds
from scipy.ndimage import label, maximum_filter, minimum_filter, uniform_filter
from shapely.geometry import shape


PROJECT_DIR = REPO_DIR
OUT_DIR = PROJECT_DIR / 'results/mars_analog'
TOOLS_DIR = PROJECT_DIR / 'analog'
CATALOGUE = PROJECT_DIR / 'results/grand_canyon/grand_canyon_detections.gpkg'
D_EARTH, D_MARS = 1229, 9237
EARTH_PIXEL_M = 200
MARS_PIXEL_M = EARTH_PIXEL_M * D_MARS / D_EARTH
MARS_RADIUS_M = 3396190
STANDARD_LAT = -8.0
RULES = json.loads((OUT_DIR / 'frozen_rules.json').read_text())
FLOOR_OPTIONS_D = [0.80, 0.90, 0.95]

sys.path.insert(0, str(TOOLS_DIR))
import flowtools  # version saved by Step B4 (supports sinks)


def px(value_D, D, pixel_m):
    return max(1, int(round(value_D * D / pixel_m)))


def closing_depth(dem, window_px):
    size = window_px | 1
    surface = minimum_filter(maximum_filter(dem, size=size), size=size)
    surface = uniform_filter(surface, size=max(3, size // 4))
    return np.clip(surface - dem, 0, None)


def share(a, b):
    return (a & b).sum() / max(a.sum(), 1)


def prepare(dem, main_depth, zone, D, pixel_m):
    """Everything that does not depend on the floor rule (computed once per planet)."""
    trunk = maximum_filter(minimum_filter(((main_depth >= RULES['trunk_depth_D'] * D) & zone).astype(np.uint8), size=5), size=5)
    labels, _ = label(trunk)
    sizes = np.bincount(labels.ravel())
    sizes[0] = 0
    trunk = labels == sizes.argmax()
    first = flowtools.fill_depressions(dem)
    receiver = flowtools.flow_receivers(first)
    stem = flowtools.main_stem(flowtools.accumulation(first, receiver), receiver, trunk)
    strip = maximum_filter(stem.astype(np.uint8), size=2 * px(RULES['floor_half_width_D'], D, pixel_m) + 1).astype(bool)
    local = closing_depth(dem, 2 * px(RULES['tributary_window_D'] / 2, D, pixel_m) + 1)
    return {'dem': dem, 'main_depth': main_depth, 'zone': zone, 'trunk': trunk, 'strip': strip, 'local': local,
            'D': D, 'pixel_m': pixel_m}


def tributaries_for(p, floor_D):
    D, pixel_m = p['D'], p['pixel_m']
    deep, _ = label((p['main_depth'] >= floor_D * D) & p['trunk'])
    deep_area = np.bincount(deep.ravel()) * pixel_m ** 2
    deep_area[0] = 0
    floor = np.isin(deep, np.flatnonzero(deep_area >= D * D)) | p['strip']
    filled = flowtools.fill_depressions(p['dem'], sinks=floor)
    receiver = flowtools.flow_receivers(filled)
    entry = flowtools.entry_points(filled, receiver, floor)
    canyon = ((p['local'] >= RULES['rim_depth_D'] * D) | (p['main_depth'] >= RULES['trunk_depth_D'] * D)) \
        & p['zone'] & ~floor & (entry >= 0)
    ids, inverse = np.unique(entry[canyon], return_inverse=True)
    area = np.bincount(inverse) * pixel_m ** 2
    incision = np.zeros(len(ids))
    np.maximum.at(incision, inverse, p['local'][canyon])
    keep = (area >= RULES['min_area_D2'] * D * D) & (incision >= RULES['min_incision_D'] * D)
    new_id = np.where(keep, np.cumsum(keep), 0)
    result = np.zeros(p['dem'].shape, dtype='int32')
    result[canyon] = new_id[inverse]
    return result


def areas_D2(result, p):
    return np.bincount(result.ravel())[1:] * p['pixel_m'] ** 2 / p['D'] ** 2


# Earth
print('Earth: preparing...')
with rasterio.open(OUT_DIR / 'grand_canyon_dem_200m.tif') as src:
    earth_transform, earth_crs = src.transform, src.crs
    earth_dem = src.read(1).astype('float64')
with rasterio.open(OUT_DIR / 'grand_canyon_incision_depth.tif') as src:
    earth_depth = src.read(1)
catalogue = gpd.read_file(CATALOGUE).to_crs('EPSG:32612')
earth_zone = rasterize([(catalogue.buffer(5000).union_all(), 1)], out_shape=earth_dem.shape, transform=earth_transform, dtype='uint8').astype(bool)
reference = rasterize([(g, 1) for g in catalogue.geometry], out_shape=earth_dem.shape, transform=earth_transform, dtype='uint8').astype(bool)
earth = prepare(earth_dem, earth_depth, earth_zone, D_EARTH, EARTH_PIXEL_M)

# Mars
print('Mars: preparing...')
metres_per_degree = np.pi / 180 * MARS_RADIUS_M
mars_x = lambda lon: lon * metres_per_degree * np.cos(np.radians(STANDARD_LAT))
bounds = (mars_x(-92.0), -18.0 * metres_per_degree, mars_x(-52.0), 0.0)
scale = MARS_PIXEL_M / 200
rasters = {}
for key, file_name in [('dem', 'valles_marineris_dem_200m.tif'), ('depth', 'valles_marineris_incision_depth.tif')]:
    with rasterio.open(OUT_DIR / file_name) as src:
        window = from_bounds(*bounds, transform=src.transform).round_offsets().round_lengths()
        shape_out = (int(window.height / scale), int(window.width / scale))
        rasters[key] = src.read(1, window=window, out_shape=shape_out, resampling=Resampling.average).astype('float64')
        mars_transform = src.window_transform(window) * Affine.scale(window.width / shape_out[1], window.height / shape_out[0])
mars = prepare(rasters['dem'], rasters['depth'], np.ones(rasters['dem'].shape, dtype=bool), D_MARS, MARS_PIXEL_M)

rows, results = [], {}
for floor_D in FLOOR_OPTIONS_D:
    print(f'Testing floor depth {floor_D} D...')
    e = tributaries_for(earth, floor_D)
    m = tributaries_for(mars, floor_D)
    precision, recall = share(e > 0, reference), share(reference, e > 0)
    ea, ma = areas_D2(e, earth), areas_D2(m, mars)
    rows.append({'floor_depth_D': floor_D, 'earth_count': len(ea),
                 'earth_precision_%': round(precision * 100), 'earth_recall_%': round(recall * 100),
                 'earth_F1_%': round(2 * precision * recall / max(precision + recall, 1e-9) * 100),
                 'earth_median_D2': round(float(np.median(ea)), 2), 'mars_count': len(ma),
                 'mars_median_D2': round(float(np.median(ma)), 2)})
    results[floor_D] = (e, m)

table = pd.DataFrame(rows)
table.to_csv(OUT_DIR / 'b4b_floor_sensitivity.csv', index=False)
best = float(table.loc[table['earth_F1_%'].idxmax(), 'floor_depth_D'])
print('\n===== Floor-rule calibration (chosen on Earth only) =====')
print('Reference before the floor rule (B3c): 233 tributaries | precision 39% | recall 76% | F1 52% | median 3.6 D2')
print(table.to_string(index=False))
print(f'Chosen floor depth: {best} D (best Earth F1). Mars uses the same value.')

RULES['floor_depth_D'] = best
(OUT_DIR / 'frozen_rules.json').write_text(json.dumps(RULES, indent=2))


def save(result, transform, crs, D, name, file_name):
    records = [{'tributary_id': int(v), 'geometry': shape(g)} for g, v in shapes(result, mask=result > 0, transform=transform)]
    out = gpd.GeoDataFrame(records, geometry='geometry', crs=crs).dissolve(by='tributary_id').reset_index()
    out['area_km2'] = out.area / 1e6
    out['area_D2'] = out.area / D ** 2
    out['planet'] = name
    out.to_file(OUT_DIR / file_name, driver='GPKG')


e, m = results[best]
save(e, earth_transform, earth_crs, D_EARTH, 'Earth', 'earth_tributaries_frozen.gpkg')
save(m, mars_transform, None, D_MARS, 'Mars', 'mars_tributaries_frozen.gpkg')
print('Saved final tributaries for both planets to:', OUT_DIR)


## B5 – Compare tributary shape on Earth and Mars

In [ ]:
# Step B5: compare the shape of tributary canyons on Earth and Mars (scale-free variables + statistics)
import json
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
from affine import Affine
from rasterio.enums import Resampling
from rasterio.features import rasterize
from rasterio.windows import from_bounds
from scipy.ndimage import maximum_filter, minimum_filter, uniform_filter
from scipy.stats import ks_2samp, mannwhitneyu
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score


OUT_DIR = REPO_DIR / 'results/mars_analog'
RULES = json.loads((OUT_DIR / 'frozen_rules.json').read_text())
D_EARTH, D_MARS = 1229, 9237
EARTH_PIXEL_M = 200
MARS_PIXEL_M = EARTH_PIXEL_M * D_MARS / D_EARTH
MARS_RADIUS_M = 3396190
STANDARD_LAT = -8.0
VARIABLES = {
    'area_D2': 'Area / D²',
    'length_D': 'Length / D',
    'width_D': 'Mean width / D',
    'elongation': 'Length / width',
    'incision_D': 'Max incision / D',
    'relief_ratio': 'Relief / length',
    'solidity': 'Solidity (1 = simple, low = branched)',
}


def closing_depth(dem, window_px):
    size = window_px | 1
    surface = minimum_filter(maximum_filter(dem, size=size), size=size)
    surface = uniform_filter(surface, size=max(3, size // 4))
    return np.clip(surface - dem, 0, None)


def measure(table, dem, transform, D, pixel_m):
    window_px = 2 * max(1, round(RULES['tributary_window_D'] / 2 * D / pixel_m)) + 1
    local = closing_depth(dem, window_px)
    ids = rasterize([(g, i + 1) for i, g in enumerate(table.geometry)], out_shape=dem.shape,
                    transform=transform, dtype='int32')
    n = len(table) + 1
    high = np.full(n, -np.inf)
    low = np.full(n, np.inf)
    deep = np.zeros(n)
    inside = ids > 0
    np.maximum.at(high, ids[inside], dem[inside])
    np.minimum.at(low, ids[inside], dem[inside])
    np.maximum.at(deep, ids[inside], local[inside])
    rows = []
    for i, geometry in enumerate(table.geometry, start=1):
        x, y = geometry.minimum_rotated_rectangle.exterior.coords.xy
        length = max(np.hypot(x[1] - x[0], y[1] - y[0]), np.hypot(x[2] - x[1], y[2] - y[1]))
        width = geometry.area / length
        rows.append({
            'area_D2': geometry.area / D ** 2,
            'length_D': length / D,
            'width_D': width / D,
            'elongation': length / width,
            'incision_D': deep[i] / D,
            'relief_ratio': (high[i] - low[i]) / length if np.isfinite(high[i]) else np.nan,
            'solidity': geometry.area / geometry.convex_hull.area,
        })
    return pd.DataFrame(rows)


# Earth
earth_table = gpd.read_file(OUT_DIR / 'earth_tributaries_frozen.gpkg')
with rasterio.open(OUT_DIR / 'grand_canyon_dem_200m.tif') as src:
    earth = measure(earth_table, src.read(1).astype('float64'), src.transform, D_EARTH, EARTH_PIXEL_M)
earth['planet'] = 'Earth'

# Mars (same grid as Step B4b)
mars_table = gpd.read_file(OUT_DIR / 'mars_tributaries_frozen.gpkg')
metres_per_degree = np.pi / 180 * MARS_RADIUS_M
mars_x = lambda lon: lon * metres_per_degree * np.cos(np.radians(STANDARD_LAT))
bounds = (mars_x(-92.0), -18.0 * metres_per_degree, mars_x(-52.0), 0.0)
scale = MARS_PIXEL_M / 200
with rasterio.open(OUT_DIR / 'valles_marineris_dem_200m.tif') as src:
    window = from_bounds(*bounds, transform=src.transform).round_offsets().round_lengths()
    shape_out = (int(window.height / scale), int(window.width / scale))
    mars_dem = src.read(1, window=window, out_shape=shape_out, resampling=Resampling.average).astype('float64')
    mars_transform = src.window_transform(window) * Affine.scale(window.width / shape_out[1], window.height / shape_out[0])
mars = measure(mars_table, mars_dem, mars_transform, D_MARS, MARS_PIXEL_M)
mars['planet'] = 'Mars'

data = pd.concat([earth, mars], ignore_index=True).dropna()
data.to_csv(OUT_DIR / 'b5_morphometry.csv', index=False)


def compare(subset, label):
    e, m = subset[subset.planet == 'Earth'], subset[subset.planet == 'Mars']
    rows = []
    for column, name in VARIABLES.items():
        u, p_mw = mannwhitneyu(e[column], m[column])
        _, p_ks = ks_2samp(e[column], m[column])
        rows.append({'variable': name, 'Earth median': round(e[column].median(), 3), 'Mars median': round(m[column].median(), 3),
                     'Mars/Earth': round(m[column].median() / e[column].median(), 2),
                     "Cliff's delta": round(2 * u / (len(e) * len(m)) - 1, 2),
                     'p (Mann-Whitney)': f'{p_mw:.3f}', 'p (KS)': f'{p_ks:.3f}'})
    features = subset[list(VARIABLES)].to_numpy()
    target = (subset.planet == 'Mars').to_numpy()
    model = RandomForestClassifier(n_estimators=300, min_samples_leaf=5, random_state=0)
    auc = cross_val_score(model, features, target, cv=StratifiedKFold(5, shuffle=True, random_state=0), scoring='roc_auc')
    print(f'\n===== {label}: Earth n={len(e)}, Mars n={len(m)} =====')
    print(pd.DataFrame(rows).to_string(index=False))
    print(f'Can a classifier tell Earth from Mars using all shape variables? ROC AUC = {auc.mean():.2f} ± {auc.std():.2f}')
    print('  (0.5 = shapes indistinguishable, 1.0 = completely different)')
    return pd.DataFrame(rows), auc.mean()


main, auc_all = compare(data, 'All tributaries')
robust, auc_big = compare(data[data.area_D2 >= 2], 'Robustness: only tributaries >= 2 D² (drops small wall facets)')
main.to_csv(OUT_DIR / 'b5_comparison.csv', index=False)

fig, axes = plt.subplots(2, 4, figsize=(20, 9))
for axis, (column, name) in zip(axes.ravel(), VARIABLES.items()):
    values = [data.loc[data.planet == p, column] for p in ('Earth', 'Mars')]
    log = column in ('area_D2', 'length_D', 'width_D', 'elongation', 'relief_ratio')
    bins = np.logspace(np.log10(max(min(v.min() for v in values), 1e-3)), np.log10(max(v.max() for v in values)), 25) if log else 25
    axis.hist(values[0], bins=bins, alpha=0.6, density=True, color='#1d4e89', label='Earth (Grand Canyon)')
    axis.hist(values[1], bins=bins, alpha=0.6, density=True, color='#d9734e', label='Mars (Valles Marineris)')
    if log:
        axis.set_xscale('log')
    axis.set_title(name)
axes.ravel()[-1].axis('off')
axes.ravel()[0].legend()
fig.suptitle(f'Tributary canyon shape, Earth vs Mars (scale-free). Classifier AUC = {auc_all:.2f}', fontsize=14)
plt.tight_layout()
plt.savefig(OUT_DIR / 'b5_shape_comparison.png', dpi=100)
plt.show()
print('Saved to:', OUT_DIR)


## B6 – Robustness checks

In [ ]:
# Step B6: is the "shallower on Mars" result real? Three robustness checks
import json
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
import rasterio
from affine import Affine
from rasterio.enums import Resampling
from rasterio.features import rasterize
from rasterio.windows import from_bounds
from scipy.ndimage import maximum_filter, minimum_filter, uniform_filter
from scipy.stats import mannwhitneyu
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split


OUT_DIR = REPO_DIR / 'results/mars_analog'
RULES = json.loads((OUT_DIR / 'frozen_rules.json').read_text())
D_EARTH, D_MARS = 1229, 9237
EARTH_PIXEL_M = 200
MARS_PIXEL_M = EARTH_PIXEL_M * D_MARS / D_EARTH
MARS_RADIUS_M = 3396190
STANDARD_LAT = -8.0
PLAN = ['area_D2', 'length_D', 'width_D', 'elongation', 'solidity']
VERTICAL = ['incision_D', 'relief_ratio']


def closing_depth(dem, window_px):
    size = window_px | 1
    surface = minimum_filter(maximum_filter(dem, size=size), size=size)
    surface = uniform_filter(surface, size=max(3, size // 4))
    return np.clip(surface - dem, 0, None)


def measure(table, dem, main_depth, transform, D, pixel_m):
    one_D = max(1, round(D / pixel_m))
    local = closing_depth(dem, 2 * max(1, round(RULES['tributary_window_D'] / 2 * D / pixel_m)) + 1)
    nearby_trunk = maximum_filter(main_depth, size=2 * one_D + 1)  # deepest main-canyon point within 1 D
    ids = rasterize([(g, i + 1) for i, g in enumerate(table.geometry)], out_shape=dem.shape, transform=transform, dtype='int32')
    n = len(table) + 1
    high, low, deep, trunk = np.full(n, -np.inf), np.full(n, np.inf), np.zeros(n), np.zeros(n)
    inside = ids > 0
    np.maximum.at(high, ids[inside], dem[inside])
    np.minimum.at(low, ids[inside], dem[inside])
    np.maximum.at(deep, ids[inside], local[inside])
    np.maximum.at(trunk, ids[inside], nearby_trunk[inside])
    rows = []
    for i, geometry in enumerate(table.geometry, start=1):
        if not np.isfinite(high[i]):
            continue
        x, y = geometry.minimum_rotated_rectangle.exterior.coords.xy
        length = max(np.hypot(x[1] - x[0], y[1] - y[0]), np.hypot(x[2] - x[1], y[2] - y[1]))
        centre = geometry.representative_point()
        rows.append({'area_D2': geometry.area / D ** 2, 'length_D': length / D, 'width_D': geometry.area / length / D,
                     'elongation': length ** 2 / geometry.area, 'solidity': geometry.area / geometry.convex_hull.area,
                     'incision_D': deep[i] / D, 'relief_ratio': (high[i] - low[i]) / length,
                     'incision_local': deep[i] / max(trunk[i], 1), 'x_km': centre.x / 1000})
    return pd.DataFrame(rows)


with rasterio.open(OUT_DIR / 'grand_canyon_dem_200m.tif') as src:
    earth_dem, earth_transform = src.read(1).astype('float64'), src.transform
with rasterio.open(OUT_DIR / 'grand_canyon_incision_depth.tif') as src:
    earth_depth = src.read(1).astype('float64')
earth = measure(gpd.read_file(OUT_DIR / 'earth_tributaries_frozen.gpkg'), earth_dem, earth_depth, earth_transform, D_EARTH, EARTH_PIXEL_M)
earth['planet'] = 'Earth'

metres_per_degree = np.pi / 180 * MARS_RADIUS_M
mars_x = lambda lon: lon * metres_per_degree * np.cos(np.radians(STANDARD_LAT))
bounds = (mars_x(-92.0), -18.0 * metres_per_degree, mars_x(-52.0), 0.0)
scale = MARS_PIXEL_M / 200
grids = {}
for key, file_name in [('dem', 'valles_marineris_dem_200m.tif'), ('depth', 'valles_marineris_incision_depth.tif')]:
    with rasterio.open(OUT_DIR / file_name) as src:
        window = from_bounds(*bounds, transform=src.transform).round_offsets().round_lengths()
        shape_out = (int(window.height / scale), int(window.width / scale))
        grids[key] = src.read(1, window=window, out_shape=shape_out, resampling=Resampling.average).astype('float64')
        mars_transform = src.window_transform(window) * Affine.scale(window.width / shape_out[1], window.height / shape_out[0])
mars = measure(gpd.read_file(OUT_DIR / 'mars_tributaries_frozen.gpkg'), grids['dem'], grids['depth'], mars_transform, D_MARS, MARS_PIXEL_M)
mars['planet'] = 'Mars'
data = pd.concat([earth, mars], ignore_index=True)
target = (data.planet == 'Mars').to_numpy()


def auc(columns):
    model = RandomForestClassifier(n_estimators=300, min_samples_leaf=5, random_state=0)
    scores = cross_val_score(model, data[columns].to_numpy(), target, cv=StratifiedKFold(5, shuffle=True, random_state=0), scoring='roc_auc')
    return f'{scores.mean():.2f} ± {scores.std():.2f}'


def cliff(a, b):
    u, p = mannwhitneyu(a, b)
    return 2 * u / (len(a) * len(b)) - 1, p


print('===== Check 1: which variables separate Earth from Mars? =====')
print('Classifier AUC, plan-view shape only (area, length, width, elongation, solidity):', auc(PLAN))
print('Classifier AUC, vertical only (incision, relief/length):                     ', auc(VERTICAL))
print('Classifier AUC, all variables:                                               ', auc(PLAN + VERTICAL))
train_x, test_x, train_y, test_y = train_test_split(data[PLAN + VERTICAL], target, test_size=0.3, stratify=target, random_state=0)
model = RandomForestClassifier(n_estimators=300, min_samples_leaf=5, random_state=0).fit(train_x, train_y)
importance = permutation_importance(model, test_x, test_y, scoring='roc_auc', n_repeats=30, random_state=0)
print(pd.Series(importance.importances_mean, index=PLAN + VERTICAL).sort_values(ascending=False).round(3).to_string())

print('\n===== Check 2: incision measured against the NEARBY main-canyon depth, not the global D =====')
for column, name in [('incision_D', 'incision / global D'), ('incision_local', 'incision / nearby main-canyon depth')]:
    e, m = data.loc[~target, column], data.loc[target, column]
    delta, p = cliff(e, m)
    print(f'{name:<40} Earth {e.median():.3f} | Mars {m.median():.3f} | Mars/Earth {m.median() / e.median():.2f} | '
          f"Cliff's delta {delta:.2f} | p {p:.4f}")

print('\n===== Check 3: is it true along the whole Valles Marineris? (Mars split west / centre / east) =====')
edges = np.quantile(mars.x_km, [0, 1 / 3, 2 / 3, 1])
earth_median = earth.incision_D.median()
for name, low, high in [('West (Ius side)', edges[0], edges[1]), ('Centre (Melas/Candor)', edges[1], edges[2]),
                        ('East (Coprates side)', edges[2], edges[3] + 1)]:
    part = mars[(mars.x_km >= low) & (mars.x_km < high)]
    delta, p = cliff(earth.incision_D, part.incision_D)
    print(f'{name:<22} n={len(part):>3} | incision/D median {part.incision_D.median():.3f} (Earth {earth_median:.3f}) | '
          f"Cliff's delta {delta:.2f} | p {p:.4f}")

data.to_csv(OUT_DIR / 'b6_robustness_data.csv', index=False)
print('\nSaved to:', OUT_DIR)
